In [ ]:
import os
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Set rendering backend based on platform
# macOS: use glfw (native OpenGL)
# Linux: use osmesa (software rendering) or egl (headless GPU)
if sys.platform == "darwin":
    os.environ["MUJOCO_GL"] = "glfw"
    print("Using macOS with GLFW rendering")
else:
    # Change this to egl if GPU is available on Linux
    os.environ["MUJOCO_GL"] = "osmesa"
    print("Using Linux with osmesa rendering")

import mediapy as media
from pathlib import Path

from track_mjx.agent import checkpointing
from track_mjx.analysis import rollout, render

import huggingface_hub as hf_hub

In [2]:
hf_checkpoint_path = "rodent/rodent-analysis/251006_144548_202519"
model_local_dir = Path.cwd().parent / "model_checkpoints"
# Download model from model repo
model_download_dir = hf_hub.snapshot_download(
    repo_id="talmolab/MIMIC-MJX", 
    repo_type="model", # download from model repo
    allow_patterns=hf_checkpoint_path + "/*", # path with model id
    local_dir=model_local_dir
)
print(f"Downloaded model to {model_download_dir}")

/n/holylabs-olveczky/Users/charleszhang/conda/mimic-mjx4/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 25 files: 100%|██████████| 25/25 [00:00<00:00, 1104.04it/s]

Downloaded model to /n/holylabs-olveczky/Users/charleszhang/track-mjx/model_checkpoints


In [3]:
hf_data_path = "data/rodent/rodent_reference_clips.h5"
# Download data from dataset repo
data_download_dir = hf_hub.hf_hub_download(
    repo_id="talmolab/MIMIC-MJX", 
    repo_type="dataset", # download from dataset repo
    filename=hf_data_path, # dataset name
    local_dir=Path.cwd().parent
)
print(f"Downloaded data to {data_download_dir}")

Downloaded data to /n/holylabs-olveczky/Users/charleszhang/track-mjx/data/rodent/rodent_reference_clips.h5


In [4]:
# replace with your checkpoint path
ckpt_path = model_local_dir / hf_checkpoint_path

ckpt = checkpointing.load_checkpoint_for_eval(ckpt_path)
cfg = ckpt["cfg"]

# Update the data path to the downloaded data
cfg.data_path = Path(data_download_dir)

Loading checkpoint from /n/holylabs-olveczky/Users/charleszhang/track-mjx/model_checkpoints/rodent/rodent-analysis/251006_144548_202519 at step 99


In [5]:
#TEMP: add max_start_frames to cfg to support older checkpoints
cfg.env_config.env_args.max_start_frame = 40

In [6]:
env = rollout.create_environment(cfg)
inference_fn = checkpointing.load_inference_fn(cfg, ckpt["policy"])
generate_rollout = rollout.create_rollout_generator(
    cfg, 
    env, 
    inference_fn, 
    log_activations=False, 
    log_metrics=False, 
    log_sensor_data=False
)

Converting to torque actuators
Rescaling body tree with scale factor 0.9


/n/holylabs-olveczky/Users/charleszhang/conda/mimic-mjx4/lib/python3.12/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [7]:
single_rollout = generate_rollout(clip_idx=0)

In [ ]:
frames, realtime_framerate = render.render_rollout(
    cfg, 
    single_rollout, 
    height=480,
    width=640,
)

# save the video to disk
media.write_video(Path(ckpt_path) / "rollout.mp4", frames, fps=realtime_framerate)
media.show_video(frames, fps=realtime_framerate)